## **Function Calling**

### **Loading genai**

In [19]:
import requests
from dotenv import load_dotenv
from google.genai import Client, types

In [2]:
load_dotenv()  # Load environment variables from .env file

client = Client()  # Initialize the Google GenAI client

- **Take Actions**: ```Interact with external systems using APIs, such as scheduling appointments, creating invoices, sending emails, or controlling smart home devices.```

- **Augment Knowledge**: ```Access information from external sources like databases, APIs, and knowledge bases.```

- **Extend Capabilities**: ```Use external tools to perform computations and extend the limitations of the model, such as using a calculator or creating charts.```

### **Augment knowledge**

In [20]:
personal_identifiers = {
    "type": "function",
    "name": "personal_identifiers",
    "description": "Extracts personal identifiers from the input text. (Name, age, hobby, education)",
    "parameters": {
        'type': 'object',
        'properties': {
            "name": {
                "type": "string", "description": "The name of the person. e.g., John Doe"
            },
            "age": {
                "type": "integer", "description": "The age of the person. e.g., 25"
            },
            "hobby": {
                "type": "string", "description": "The hobby of the person. e.g., reading, swimming, painting"
            },
            "education": {
                "type": "string", "description": "The education level of the person. e.g., bachelor's degree, master's degree"
            }
        },
        'required': ['name', 'age', 'hobby', 'education']
    }
}

In [25]:
interactions = client.interactions.create(
    model="gemini-3-flash-preview",
    input="Hello everyone, my name is John Doe. I am 25 years old and I love reading books. I have a bachelor's degree in Computer Science.",
    tools=[personal_identifiers]
)

In [27]:
for i in interactions:
    print(i)

('status', 'requires_action')
('model', 'gemini-3-flash-preview')
('agent', None)
('id', 'v1_ChdEVnRXYXRlWERQYmpuc0VQNGJMdS1BMBIXRFZ0V2F0ZVhEUGJqbnNFUDRiTHUtQTA')
('created', '2026-07-14T15:51:41Z')
('updated', '2026-07-14T15:51:41Z')
('system_instruction', None)
('tools', None)
('usage', Usage(cached_tokens_by_modality=None, grounding_tool_count=None, input_tokens_by_modality=[ModalityTokens(modality='text', tokens=205)], output_tokens_by_modality=None, tool_use_tokens_by_modality=None, total_cached_tokens=0, total_input_tokens=205, total_output_tokens=42, total_thought_tokens=122, total_tokens=369, total_tool_use_tokens=0))
('response_modalities', None)
('response_mime_type', None)
('previous_interaction_id', None)
('environment_id', None)
('service_tier', 'standard')
('webhook_config', None)
('steps', [FunctionCallStep(arguments={'age': 25, 'name': 'John Doe', 'hobby': 'reading books', 'education': "bachelor's degree in Computer Science"}, id='qpjmemq7', name='personal_identifiers',

In [21]:
inputs = [
    "Hello everyone, my name is John Doe. I am 25 years old and I love reading books. I have a bachelor's degree in Computer Science.",
    "Hi, I'm Jane Smith. I'm 30 years old and my favorite hobby is painting. I hold a master's degree in Fine Arts.",
    "Greetings! My name is Michael Johnson. I'm 28 years old and I enjoy swimming. I have a bachelor's degree in Mechanical Engineering.",
    "Hey there, I'm Emily Davis. I'm 22 years old and I love hiking. I have a bachelor's degree in Environmental Science.",
    "Hello, I'm David Wilson. I'm 35 years old and my hobby is playing guitar. I hold a master's degree in Music."
]

In [28]:
results = []

for input_text in inputs:
    response = client.interactions.create(
        model="gemini-3-flash-preview",
        input=input_text,
        tools=[personal_identifiers]
    )

    fc_step = next(s for s in response.steps if s.type == "function_call")

    if fc_step.name == "personal_identifiers":
        result = fc_step.arguments
        results.append(result)

In [31]:
results

[{'age': 25,
  'education': "bachelor's degree in Computer Science",
  'hobby': 'reading books',
  'name': 'John Doe'},
 {'name': 'Jane Smith',
  'age': 30,
  'education': "master's degree in Fine Arts",
  'hobby': 'painting'},
 {'age': 28,
  'hobby': 'swimming',
  'name': 'Michael Johnson',
  'education': "bachelor's degree in Mechanical Engineering"},
 {'hobby': 'hiking',
  'age': 22,
  'education': "bachelor's degree in Environmental Science",
  'name': 'Emily Davis'},
 {'hobby': 'playing guitar',
  'name': 'David Wilson',
  'age': 35,
  'education': "master's degree in Music"}]

In [30]:
import pandas as pd

df = pd.DataFrame(results)
df

,age,education,hobby,name
0,25,bachelor's degree in Computer Science,reading books,John Doe
1,30,master's degree in Fine Arts,painting,Jane Smith
2,28,bachelor's degree in Mechanical Engineering,swimming,Michael Johnson
3,22,bachelor's degree in Environmental Science,hiking,Emily Davis
4,35,master's degree in Music,playing guitar,David Wilson


In [32]:
df.to_csv('personal_identifiers_results.csv', index=False)

### **Example 2**

In [4]:
mul_fn_dec = {
    "type": "function",
    "name": "mul",
    "description": "Multiplies two numbers and returns the result.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "The first number."},
            "b": {"type": "number", "description": "The second number."}
        },
        "required": ["a", "b"]
    }
}

# tools = types.Tool(function_declaration=[mul_fn_dec])
# config = types.GenerateContentConfig(tools=[tools])

def mul(a: float, b: float) -> float:
    """Multiplies two numbers and returns the result."""
    return a * b

In [7]:
input = "What is the multiplication of 3534 and 3245?"

response = client.interactions.create(
    model="gemini-3.5-flash",
    input=input,
    # tools=[mul_fn_dec]
)

In [9]:
print(response.output_text)  # Display the model's response

The multiplication of 3,534 and 3,245 is **11,467,830**. 

Here is the step-by-step breakdown:
* 3,534 × 5 = 17,670
* 3,534 × 40 = 141,360
* 3,534 × 200 = 706,800
* 3,534 × 3,000 = 10,602,000

**Total:** 17,670 + 141,360 + 706,800 + 10,602,000 = **11,467,830**


In [10]:
response = client.interactions.create(
    model="gemini-3.5-flash",
    input=input,
    tools=[mul_fn_dec]
)

response.output_text

''

In [11]:
set_light_values_declaration = {
    "type": "function",
    "name": "set_light_values",
    "description": "Sets the brightness and color temperature of a light.",
    "parameters": {
        "type": "object",
        "properties": {
            "brightness": {
                "type": "integer",
                "description": "Light level from 0 to 100",
            },
            "color_temp": {
                "type": "string",
                "enum": ["daylight", "cool", "warm"],
                "description": "Color temperature",
            },
        },
        "required": ["brightness", "color_temp"],
    },
}

def set_light_values(brightness: int, color_temp: str) -> dict:
    """Set the brightness and color temperature of a room light."""
    return {"brightness": brightness, "colorTemperature": color_temp}

In [12]:
from google import genai

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.5-flash",
    input="Turn the lights down to a romantic level",
    tools=[set_light_values_declaration],
)

fc_step = next(s for s in interaction.steps if s.type == "function_call")
print(fc_step)

arguments={'brightness': 20, 'color_temp': 'warm'} id='bwfji93h' name='set_light_values' type='function_call' signature='Er4ICrsIARFNMg+mNcASHRE0UdnV/YpW8x8y3J3WJ5RORpJVo4VvS67RKBxQLLQhG3OuOKZel0VL8kHbL1WH4cGxHLVkDqgftisNsOrzb+UL3SA1xROD3FoZC9HaR8Hj9/k9qEyWdHT8KQQrglBnnzvymFNfUs4/+s0Ip9MfN2v5DtRDc0jPJYomgpytscvsDWV/XRDgfPZW7s2AZlNh7jQ5H+3N3CkcWQEAplmocAqTy9rqll9vqXaE6uzu6fb/Gps51jBw18CfO3WKoa2ssZTY+YLwoLCmYdS4c7aOprCzXo1ttukWJHswsYvohkegn3Bn9d3fV9ZOC9iFrFBhen2xdEAmZWhv0F5FeIyNH9CdXIG0gigFhnq2uTglxm/zLgd5Mr+IsUPZSkOAgAIugkbJYWeSDO6s2ijF1zvlBSQ1NbN956DqmreVWA0ycYZ5PAMiwNewfnOKMsrC3B953K83hsSCrtDlYS6FnqL0uB7W60HxlZzr3iqmoPpoFA9wqTl5rt8seiOcx3ncHXhXFo4Nt0yYl9GYqJ7F/9HouH3gfTA+0epuVwwpU6oUHCBsJnlhN4qunDWC6NkHnfVn/X1N7AXMdzfxwboKVGqpaP+WK4zPfcXncomWHvQer+Ko0KoOW0rq4GghdiH+RALF3yD1S2WI+34uMQGWfoP2OimBSNCBXbH6gCT5UT90nZ0J76eTh8bawKGQPnEfFuzSMjrsgl4MkCsZDW9wnG4dvVVZQBdHeJMwnxCOxuGQ1i5XdVCmOfsnmktDF84N1Vp83lO8pyKzRDYyXBLGmq1RmsByXqgNCube2xh89p2YBfT0LZKmi4BENe9hAsFbV/V7O9l+TK2lHE0C9nCsbpI/gBbhy/I

In [17]:
for i in fc_step:
    print(i)

('arguments', {'brightness': 20, 'color_temp': 'warm'})
('id', 'bwfji93h')
('name', 'set_light_values')
('type', 'function_call')
('signature', 'Er4ICrsIARFNMg+mNcASHRE0UdnV/YpW8x8y3J3WJ5RORpJVo4VvS67RKBxQLLQhG3OuOKZel0VL8kHbL1WH4cGxHLVkDqgftisNsOrzb+UL3SA1xROD3FoZC9HaR8Hj9/k9qEyWdHT8KQQrglBnnzvymFNfUs4/+s0Ip9MfN2v5DtRDc0jPJYomgpytscvsDWV/XRDgfPZW7s2AZlNh7jQ5H+3N3CkcWQEAplmocAqTy9rqll9vqXaE6uzu6fb/Gps51jBw18CfO3WKoa2ssZTY+YLwoLCmYdS4c7aOprCzXo1ttukWJHswsYvohkegn3Bn9d3fV9ZOC9iFrFBhen2xdEAmZWhv0F5FeIyNH9CdXIG0gigFhnq2uTglxm/zLgd5Mr+IsUPZSkOAgAIugkbJYWeSDO6s2ijF1zvlBSQ1NbN956DqmreVWA0ycYZ5PAMiwNewfnOKMsrC3B953K83hsSCrtDlYS6FnqL0uB7W60HxlZzr3iqmoPpoFA9wqTl5rt8seiOcx3ncHXhXFo4Nt0yYl9GYqJ7F/9HouH3gfTA+0epuVwwpU6oUHCBsJnlhN4qunDWC6NkHnfVn/X1N7AXMdzfxwboKVGqpaP+WK4zPfcXncomWHvQer+Ko0KoOW0rq4GghdiH+RALF3yD1S2WI+34uMQGWfoP2OimBSNCBXbH6gCT5UT90nZ0J76eTh8bawKGQPnEfFuzSMjrsgl4MkCsZDW9wnG4dvVVZQBdHeJMwnxCOxuGQ1i5XdVCmOfsnmktDF84N1Vp83lO8pyKzRDYyXBLGmq1RmsByXqgNCube2xh89p2YBfT0LZKmi4BENe9hAsFbV/V7O9l

In [15]:
fc_step = next(s for s in interaction.steps if s.type == "function_call")

if fc_step.name == "set_light_values":
    result = set_light_values(**fc_step.arguments)
    print(f"Function execution result: {result}")

Function execution result: {'brightness': 20, 'colorTemperature': 'warm'}
